# Frequency-search results explorer

Look at the output of an adversarial frequency search (`freq_search.py`): the
per-`(letter, pattern)` multipliers, the resulting per-word frequencies, and the
greedy strategy analyzed the *same way* as in `strategy_explorer` but under the
adversarial weighting (greedy playing against the adversarial word choices).

Generate a run first with e.g. `python math/freq_search.py 5` (saves to
`math/data/freq_search/len05.csv`); this notebook loads it back.

In [ ]:
from pathlib import Path
import sys

def find_math_dir():
    cwd = Path.cwd().resolve()
    for c in (cwd, cwd.parent, cwd.parent.parent, cwd / 'math', cwd.parent / 'math'):
        if (c / 'freq_search.py').exists():
            return c
    raise FileNotFoundError('Could not find freq_search.py. Open from the repo root, math/, or math/notebooks/.')

math_dir = find_math_dir()
if str(math_dir) not in sys.path:
    sys.path.insert(0, str(math_dir))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import freq_search, strategy

print(f'imported from {math_dir}')

## Load a search result

Loads the saved CSV for this length if present; otherwise runs one inline (which
can be slow for larger lengths).

In [ ]:
LENGTH = 5

csv_path = math_dir / 'data' / 'freq_search' / f'len{LENGTH:02d}.csv'
if csv_path.exists():
    s = freq_search.FrequencySearch.reconstruct(LENGTH, csv_path)
    print(f'loaded saved run: {csv_path.name}')
else:
    print(f'no saved run at {csv_path}\nrunning one inline (may take a while)...')
    s = freq_search.FrequencySearch(LENGTH)
    s.run(verbose=False)

game, f = s.game, s.freq
print(f'length {LENGTH}: {game.N} words, {s.M} patterns')
print(f'greedy avg misses -- uniform: {game.average_misses():.4f}   '
      f'adversarial: {game.average_misses(f):.4f}')

## Multipliers

One row per `(letter, pattern)`; `multiplier` is the tuned weight, `n_words` how
many words carry that pattern.

In [ ]:
mult_df = s.multipliers_df()
print(f'{(mult_df.multiplier == 0).sum()} patterns zeroed; '
      f'{(mult_df.multiplier > 1).sum()} boosted above 1; '
      f'max multiplier {mult_df.multiplier.max():g}')
mult_df.head(20)

## Frequencies

One row per word (frequency = product of its pattern multipliers).

In [ ]:
freq_df = s.frequencies_df()
print(f'{(freq_df.frequency == 0).sum()} words at frequency 0 (removed); '
      f'heaviest carry the adversarial weight:')
freq_df.head(15)

## Greedy under the adversarial weighting

The same analysis as `strategy_explorer`, but passing `freq = s.freq` so the
greedy plays against (and is scored by) the adversarial distribution.

In [ ]:
print(f'greedy opener -- uniform: {game.greedy_first_letter()!r}   '
      f'adversarial: {game.greedy_first_letter(f)!r}')

by_letter = game.misses_by_starting_letter(f)      # adversarial
mb = pd.Series(by_letter, name='avg_misses'); mb.index.name = 'first_letter'

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.bar(mb.index, mb.values, color='#a54b3b')
ax.axhline(game.average_misses(f), color='#333', ls='--', lw=1, label='greedy opener')
ax.set_ylabel('avg misses')
ax.set_title(f'Expected misses by opening letter -- adversarial (length {LENGTH})')
ax.legend()
plt.show()
mb.to_frame().T

## Per-word difficulty (adversarial)

In [ ]:
pwm = game.per_word_misses(f)
words = pd.DataFrame({'word': game.words, 'freq': f, 'misses': pwm.astype(int)})

print(f'max misses {int(pwm.max())};  frequency-weighted avg {game.average_misses(f):.4f}')
fig, ax = plt.subplots(figsize=(9, 3))
ax.hist(pwm, bins=range(0, int(pwm.max()) + 2), color='#a54b3b', align='left', rwidth=0.85)
ax.set_xlabel('misses'); ax.set_ylabel('# words'); plt.show()

print('hardest words (with their adversarial weight):')
words.sort_values('misses', ascending=False).head(15).reset_index(drop=True)

## Trace a word (adversarial)

The greedy guess sequence for a word, under the adversarial weighting.

In [ ]:
WORD = game.words[int(pwm.argmax())]      # the hardest word; or set your own

hits = game.trace(WORD, f)
n_miss = sum(1 for _, hit in hits if not hit)
print(f'{WORD!r}: {n_miss} misses')
for letter, hit in hits:
    mark = 'hit ' if hit else 'MISS'
    print(f'  {letter}  {mark}')